# Mein Zimmer als 3D-Modell (3D Gaussian Splatting auf Colab Free)

Dieses Notebook nimmt ein Video deines Zimmers aus Google Drive, schaetzt Kameraposen mit COLMAP, trainiert ein 3D Gaussian Splatting Modell mit **gsplat**, und exportiert am Ende eine `.ply` Datei die du in **SuperSplat** (Browser) oder lokalen Viewern anschauen kannst.

Optional kann anschliessend **Difix3D** zur Artefakt-Entfernung angewendet werden.

---

## Bevor du startest

1. **Runtime auf GPU stellen**: `Runtime` -> `Change runtime type` -> Hardware accelerator: **T4 GPU**
2. **Video aufnehmen** (mit dem Handy):
   - 30-60 Sekunden, **langsam** durch den Raum laufen
   - mehrere Hoehen abdecken (Augenhoehe, etwas oben/unten)
   - gutes Licht, **keine** Spiegel/Fenster mit direkter Sonne
   - der Raum sollte Textur haben (komplett weisse Waende = COLMAP scheitert)
3. **Video in Google Drive hochladen** unter z.B.:
   ```
   MyDrive/room3d/room.mp4
   ```
4. Erwartete Laufzeit auf Colab Free (T4): **ca. 1-2 Stunden** fuer Baseline 3DGS, +1h fuer optionalen Difix3D-Schritt.

---

## Schritt 1: GPU pruefen & Drive einbinden

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Schritt 2: Konfiguration

Passe hier ggf. den Pfad zu deinem Video an. Alles andere kann meistens so bleiben.

In [ ]:
import os, glob

# --- ANPASSEN ---
# Ordner (oder Eltern-Ordner), unter dem GENAU EIN Video liegt. Wird rekursiv
# durchsucht -> egal wie der Unterordner exakt heisst (auch Leerzeichen am Ende).
# "Meine Ablage" in der Drive-Oberflaeche heisst im Dateisystem immer "MyDrive".
SEARCH_ROOT = '/content/drive/MyDrive/3DSeminar Bilder/Video Zimmer /'
TARGET_FRAMES = 150                                     # ~100-200 ist gut fuer T4
DRIVE_OUTPUT = '/content/drive/MyDrive/room3d/output'   # Ergebnisse landen hier (persistent)

# --- meist nicht aendern ---
SCENE_DIR = '/content/scene'                            # lokales Arbeitsverzeichnis
DATA_FACTOR = 4                                         # downscale fuer Training (4 = 1/4 Aufloesung)
# 20000 Schritte: 8000 davon mit Densification (siehe Schritt 7), die restlichen
# 12000 verfeinern die fixierte Gaussian-Menge. Bringt deutlich bessere Out-of-
# Distribution-Views (Novel View Synthesis) als 15000.
MAX_STEPS = 20000                                       # gsplat Training Schritte

# Das eine Video rekursiv finden (egal ob .mov/.mp4/.MOV, egal welcher Unterordner)
vids = [f for f in glob.glob(os.path.join(SEARCH_ROOT, '**', '*'), recursive=True)
        if f.lower().endswith(('.mov', '.mp4', '.m4v', '.avi', '.mkv'))]
assert len(vids) == 1, f'Erwarte genau 1 Video unter {SEARCH_ROOT}, gefunden: {vids}'
VIDEO_PATH = vids[0]

os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print('Video gefunden:', VIDEO_PATH)
print('Output landet in:', DRIVE_OUTPUT)

## Schritt 3: Dependencies installieren

Dauert beim ersten Run **5-10 Minuten** (gsplat baut ggf. CUDA-Extensions).

In [ ]:
# System-Tools
!apt-get -qq install -y colmap ffmpeg > /dev/null
print('COLMAP + ffmpeg installiert.')

In [ ]:
# Python deps. gsplat hat fuer aktuelle torch/cuda Versionen Prebuilt Wheels - sonst dauert es laenger.
import torch
print('torch:', torch.__version__, '| cuda:', torch.version.cuda)

!pip install -q gsplat
!pip install -q viser nerfview tyro imageio[ffmpeg] tqdm Pillow \
    torchmetrics opencv-python pycolmap plyfile splines

# fused-ssim gibt es NICHT als PyPI-Wheel -> aus Git bauen (CUDA-Extension).
# WICHTIG: --no-build-isolation, damit gegen das bereits installierte torch gebaut
# wird (sonst zieht pip ein fremdes torch in die Build-Umgebung -> CUDA-Build schlaegt fehl).
# Hinweis: fused-ssim ist fuer gsplat OPTIONAL - faellt das Bauen aus, laeuft das
# Training trotzdem (gsplat nutzt dann seine eigene SSIM-Implementierung).
!pip install -q ninja
!pip install --no-build-isolation git+https://github.com/rahul-goel/fused-ssim.git \
    || echo 'WARNUNG: fused-ssim Build fehlgeschlagen - Training laeuft auch ohne, nur etwas langsamer.'

print('Done.')

## Schritt 4: Frames aus dem Video extrahieren (mit Schaerfe-Filter)

Wir extrahieren `OVERSAMPLE`x so viele Kandidaten-Frames wie noetig, messen pro
Frame die Schaerfe (Varianz des Laplace-Operators) und behalten **pro Zeitfenster
nur den schaerfsten**. Bewegungsunscharfe Frames werden so automatisch aussortiert
-> bessere COLMAP-Posen und schaerferes Gaussian Splatting.

In [ ]:
import subprocess, json, glob, shutil
import cv2
import numpy as np

CAND_DIR = f'{SCENE_DIR}/_candidates'   # temporaere Kandidaten
IMG_DIR  = f'{SCENE_DIR}/images'        # finale, scharfe Frames fuer COLMAP
OVERSAMPLE = 3                          # so viele Kandidaten pro Ziel-Frame extrahieren
MIN_SHARP_FRAC = 0.15                   # Frames unter 15% der Median-Schaerfe komplett verwerfen

# frisch starten
for d in (CAND_DIR, IMG_DIR):
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

# Video-Dauer bestimmen
probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
     '-of', 'json', VIDEO_PATH], capture_output=True, text=True)
duration = float(json.loads(probe.stdout)['format']['duration'])
cand_fps = (TARGET_FRAMES * OVERSAMPLE) / duration
print(f'Video-Dauer: {duration:.1f}s -> Kandidaten-FPS: {cand_fps:.2f} (Ziel: {TARGET_FRAMES} scharfe Frames)')

# 1) Viele Kandidaten extrahieren
!ffmpeg -y -i "{VIDEO_PATH}" -vf "fps={cand_fps},scale=1280:-2" \
    -q:v 2 "{CAND_DIR}/cand_%05d.jpg" -loglevel error

cands = sorted(glob.glob(f'{CAND_DIR}/cand_*.jpg'))
print(f'{len(cands)} Kandidaten extrahiert. Berechne Schaerfe (Laplacian-Varianz)...')

# 2) Schaerfe pro Kandidat (Varianz des Laplace-Operators)
def sharpness(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return 0.0
    return float(cv2.Laplacian(img, cv2.CV_64F).var())

sharp = np.array([sharpness(c) for c in cands])

# 3) In TARGET_FRAMES zeitliche Fenster aufteilen, pro Fenster den schaerfsten behalten
n_bins = min(TARGET_FRAMES, len(cands))
edges = np.linspace(0, len(cands), n_bins + 1).astype(int)
picks = []
for b in range(n_bins):
    lo, hi = edges[b], edges[b + 1]
    if hi <= lo:
        continue
    best = lo + int(np.argmax(sharp[lo:hi]))
    picks.append(best)

# 4) Extrem unscharfe zusaetzlich verwerfen (relativ zur Median-Schaerfe der Auswahl)
med = float(np.median(sharp[picks]))
thr = MIN_SHARP_FRAC * med
kept = [i for i in picks if sharp[i] >= thr]
dropped = [i for i in picks if sharp[i] < thr]

# 5) Finale Frames sauber durchnummeriert nach images/ kopieren
for j, i in enumerate(kept, start=1):
    shutil.copy(cands[i], f'{IMG_DIR}/frame_{j:04d}.jpg')

shutil.rmtree(CAND_DIR, ignore_errors=True)

print(f'\n[OK] {len(kept)} scharfe Frames -> {IMG_DIR}')
print(f'     Median-Schaerfe der Auswahl: {med:.0f} | Schwelle: {thr:.0f}')
print(f'     {len(dropped)} zu unscharfe Frames verworfen.')
if len(kept) < 0.6 * TARGET_FRAMES:
    print('     HINWEIS: Viele Frames waren unscharf -> Video war vermutlich verwackelt.'
          ' Beim naechsten Dreh langsamer schwenken.')

## Schritt 5: COLMAP - Kameraposen schaetzen

**Dauer: ~10-20 Min auf Colab CPU** (apt-COLMAP hat oft kein CUDA, daher CPU).

Wenn der `mapper` keinen `sparse/0/` Ordner erzeugt, hat dein Video zu wenig Overlap oder zu wenig Textur - dann nochmal langsamer filmen.

In [ ]:
import shutil

# COLMAP-Cache liegt persistent in Drive. Ist er vorhanden -> wird COLMAP NICHT
# neu gerechnet, sondern der fertige Stand zurueckgeholt (spart ~80 Min).
# Pfad ggf. anpassen, falls dein Video woanders liegt:
COLMAP_CACHE = '/content/drive/MyDrive/3DSeminar Bilder/Video Zimmer/colmap'
FORCE_RECOMPUTE = False                         # True = Cache ignorieren, neu rechnen

db_path    = f'{SCENE_DIR}/database.db'
sparse_dir = f'{SCENE_DIR}/sparse/0'
os.makedirs(f'{SCENE_DIR}/sparse', exist_ok=True)

# 0) Falls Cache in Drive existiert -> ins lokale Arbeitsverzeichnis zurueckholen
if (not FORCE_RECOMPUTE
        and os.path.exists(f'{COLMAP_CACHE}/sparse/0/cameras.bin')
        and not os.path.exists(f'{sparse_dir}/cameras.bin')):
    print('Gecachte COLMAP-Ergebnisse in Drive gefunden -> wiederherstellen...')
    if os.path.exists(f'{COLMAP_CACHE}/database.db'):
        shutil.copy(f'{COLMAP_CACHE}/database.db', db_path)
    shutil.copytree(f'{COLMAP_CACHE}/sparse', f'{SCENE_DIR}/sparse', dirs_exist_ok=True)

# 1) Schon vorhanden? -> teuren Lauf ueberspringen
if os.path.exists(f'{sparse_dir}/cameras.bin') and not FORCE_RECOMPUTE:
    print(f'[SKIP] COLMAP bereits vorhanden: {os.listdir(sparse_dir)}')
else:
    if FORCE_RECOMPUTE and os.path.exists(db_path):
        os.remove(db_path)

    # 1a) Feature Extraction
    !colmap feature_extractor \
        --database_path {db_path} \
        --image_path {SCENE_DIR}/images \
        --ImageReader.single_camera 1 \
        --ImageReader.camera_model OPENCV \
        --SiftExtraction.use_gpu 0

    # 1b) Matching
    !colmap exhaustive_matcher \
        --database_path {db_path} \
        --SiftMatching.use_gpu 0

    # 1c) Sparse Reconstruction (Posen)
    !colmap mapper \
        --database_path {db_path} \
        --image_path {SCENE_DIR}/images \
        --output_path {SCENE_DIR}/sparse

    if not os.path.exists(f'{sparse_dir}/cameras.bin'):
        raise RuntimeError('COLMAP konnte keine Rekonstruktion erstellen. Video hat vermutlich zu wenig Overlap/Textur.')

    # 1d) Ergebnis nach Drive sichern -> beim naechsten Mal SKIP
    os.makedirs(COLMAP_CACHE, exist_ok=True)
    shutil.copy(db_path, f'{COLMAP_CACHE}/database.db')
    if os.path.exists(f'{COLMAP_CACHE}/sparse'):
        shutil.rmtree(f'{COLMAP_CACHE}/sparse')
    shutil.copytree(f'{SCENE_DIR}/sparse', f'{COLMAP_CACHE}/sparse')
    print(f'\n[OK] COLMAP fertig + nach Drive gesichert -> {COLMAP_CACHE}')

print('Files in sparse/0/:', os.listdir(sparse_dir))

## Schritt 5b: Bilder undistorten (Linsenverzerrung rausrechnen)

iPhone-Kameras (und die meisten Handys) haben **Weitwinkel-Verzerrung** —
gerade Kanten erscheinen am Bildrand leicht gekruemmt. COLMAP modelliert das
mit dem `OPENCV`-Camera-Modell (4 Distortion-Parameter), aber **gsplat rendert
mit einer reinen Lochkamera** und ignoriert diese Parameter. Folge: das Modell
lernt eine leicht widerspruechliche Geometrie, was bei Novel-View-Synthese in
SuperSplat & Co. zu schmierigen Artefakten fuehrt.

Loesung: `colmap image_undistorter` warpt Bilder + Posen so um, dass sie einer
reinen Lochkamera entsprechen. Die Zelle ist idempotent (skippt sich selbst,
wenn das Camera-Modell schon PINHOLE ist).

In [ ]:
import shutil, struct

# Schaut auf das Camera-Modell in der ersten Camera-Definition. Solange COLMAP
# noch die OPENCV/RADIAL-Verzerrung modelliert (Modell-IDs > 1), werden Bilder
# und Posen via colmap image_undistorter auf eine reine Lochkamera (PINHOLE)
# umgerechnet. Das eliminiert den schmierigen "Schmier-Bias" bei Novel Views.
# Modell-IDs: 0=SIMPLE_PINHOLE, 1=PINHOLE, alles andere = mit Distortion.
def _first_camera_model_id(bin_path):
    with open(bin_path, 'rb') as _f:
        _f.read(8)                              # num_cameras (uint64)
        _f.read(4)                              # camera_id (uint32)
        return struct.unpack('<i', _f.read(4))[0]

cam_model = _first_camera_model_id(f'{sparse_dir}/cameras.bin')
if cam_model in (0, 1):
    print(f'[SKIP] Camera-Modell ist bereits PINHOLE/SIMPLE_PINHOLE (id={cam_model}).')
else:
    print(f'[Undistort] Camera-Modell id={cam_model} -> warpe Bilder + Posen auf PINHOLE...')
    undist_tmp = f'{SCENE_DIR}/_undist_tmp'
    if os.path.exists(undist_tmp):
        shutil.rmtree(undist_tmp)

    # max_image_size 2000: limitiert die Ausgabegroesse; iPhone-Frames sind eh
    # schon auf 1280 in Schritt 4 skaliert, also unkritisch.
    !colmap image_undistorter \
        --image_path {SCENE_DIR}/images \
        --input_path {sparse_dir} \
        --output_path {undist_tmp} \
        --output_type COLMAP \
        --max_image_size 2000

    if not os.path.exists(f'{undist_tmp}/sparse/cameras.bin'):
        raise RuntimeError('colmap image_undistorter ist gescheitert.')

    # Originale durch undistorted ersetzen.
    shutil.rmtree(f'{SCENE_DIR}/images')
    shutil.move(f'{undist_tmp}/images', f'{SCENE_DIR}/images')

    # undistorter legt sparse direkt in sparse/ (nicht sparse/0/) -> umsortieren,
    # damit gsplat's Dataloader das wiederfindet.
    shutil.rmtree(f'{SCENE_DIR}/sparse')
    os.makedirs(f'{SCENE_DIR}/sparse/0', exist_ok=True)
    for fn in os.listdir(f'{undist_tmp}/sparse'):
        shutil.move(f'{undist_tmp}/sparse/{fn}', f'{SCENE_DIR}/sparse/0/{fn}')
    shutil.rmtree(undist_tmp, ignore_errors=True)

    # Alte downscale-Ordner verwerfen, Schritt 6 baut sie aus den neuen Bildern neu.
    for f in (2, 4, 8):
        for suf in ('', '_png'):
            d = f'{SCENE_DIR}/images_{f}{suf}'
            if os.path.exists(d):
                shutil.rmtree(d)

    new_model = _first_camera_model_id(f'{sparse_dir}/cameras.bin')
    n_images = len(os.listdir(f'{SCENE_DIR}/images'))
    print(f'[OK] Undistortion fertig. Camera-Modell id={new_model} (1=PINHOLE), '
          f'{n_images} Bilder.')

## Schritt 6: Downscaled Bilder fuer Training

gsplat erwartet downscaled-Varianten der Bilder (z.B. `images_4/`).

In [ ]:
from PIL import Image
from tqdm import tqdm

src_dir = f'{SCENE_DIR}/images'
for factor in [2, 4, 8]:
    dst_dir = f'{SCENE_DIR}/images_{factor}'
    os.makedirs(dst_dir, exist_ok=True)
    for fn in tqdm(sorted(os.listdir(src_dir)), desc=f'images_{factor}'):
        img = Image.open(os.path.join(src_dir, fn))
        w, h = img.size
        img.resize((w // factor, h // factor), Image.LANCZOS).save(
            os.path.join(dst_dir, fn), quality=92)
print('Downscaled Versionen erstellt.')

import subprocess, time, shutil

# WICHTIG: Training schreibt seine Ckpts/Videos LOKAL (/content/...), NICHT direkt
# nach Drive. Gruende:
#  1) Drive-I/O ist langsam und kann waehrend Training haengen.
#  2) Bei Drive >75% voll fuehren Writes zu I/O-Errors und killen das Training.
# Am Ende der Zelle synchronisieren wir LOCAL -> Drive (auch bei Crash, damit
# Teil-Ergebnisse nicht verloren gehen). Die nachfolgenden Zellen lesen weiter
# ueber RESULT_DIR -> Drive-Pfad.
RESULT_DIR_LOCAL = '/content/gsplat_base'
RESULT_DIR       = f'{DRIVE_OUTPUT}/gsplat_base'
LOG_PATH         = '/content/train.log'
os.makedirs(RESULT_DIR_LOCAL, exist_ok=True)
os.makedirs(RESULT_DIR,       exist_ok=True)

# Training im Hintergrund + heartbeat alle 60s (nur letzte 5 Logzeilen).
# Verhindert "something went wrong displaying the page" / WebSocket-Reconnect-
# Loops durch tqdm-Output-Flut.
#
# Wichtige Trainings-Flags:
#  --strategy.refine-stop-iter 8000  -> Densification (Split/Duplicate) laeuft
#       bis Step 8000. Endet typischerweise mit ~800k-1M Gaussians, was T4 mit
#       DATA_FACTOR=4 gerade noch packt. Anschliessend verfeinern die restlichen
#       12000 Steps die fixierte Menge. Ist der Hauptdreher fuer Detail-Qualitaet
#       und Novel-View-Generalisation. Falls T4 mit OOM crasht: auf 6000 senken.
#  --disable_viewer                  -> kein Viser-Server (in Colab eh blockiert).
env = {**os.environ, 'CUDA_VISIBLE_DEVICES': '0'}
log_f = open(LOG_PATH, 'w')
proc = subprocess.Popen(
    ['python', '/content/gsplat/examples/simple_trainer.py', 'default',
     '--data_dir',    SCENE_DIR,
     '--data_factor', str(DATA_FACTOR),
     '--result_dir',  RESULT_DIR_LOCAL,
     '--max_steps',   str(MAX_STEPS),
     '--strategy.refine-stop-iter', '8000',
     '--disable_viewer'],
    stdout=log_f, stderr=subprocess.STDOUT, env=env,
)

print(f'Training laeuft (PID {proc.pid}). Volles Log: {LOG_PATH}')
print(f'Lokales Ergebnis: {RESULT_DIR_LOCAL}  (wird am Ende nach Drive kopiert)')
print('Heartbeat alle 60s mit den letzten Logzeilen...\n')

try:
    while proc.poll() is None:
        time.sleep(60)
        try:
            with open(LOG_PATH, 'rb') as f:
                raw = f.read().decode('utf-8', errors='replace')
            lines = [ln for ln in raw.replace('\r', '\n').splitlines() if ln.strip()]
            print('--- ' + time.strftime('%H:%M:%S') + f' (PID {proc.pid}) ---')
            for line in lines[-5:]:
                print(line)
            print()
        except Exception as e:
            print('  (Log nicht lesbar:', e, ')')
except KeyboardInterrupt:
    print('\nAbbruch durch User -> terminiere Training...')
    proc.terminate()
    proc.wait(timeout=30)
    raise
finally:
    log_f.close()

print('Training beendet (Exit-Code:', proc.returncode, ')')

# Ergebnisse nach Drive synchronisieren - auch wenn Training gecrasht ist,
# damit zumindest die vorhandenen Checkpoints/Videos persistent gesichert sind.
def _sync(src_root, dst_root):
    copied = 0
    for sub in ('ckpts', 'videos', 'stats'):
        src = os.path.join(src_root, sub)
        if not os.path.isdir(src):
            continue
        dst = os.path.join(dst_root, sub)
        os.makedirs(dst, exist_ok=True)
        for fn in os.listdir(src):
            sp, dp = os.path.join(src, fn), os.path.join(dst, fn)
            if os.path.isfile(sp) and (not os.path.exists(dp)
                                       or os.path.getsize(sp) != os.path.getsize(dp)):
                shutil.copy(sp, dp); copied += 1
    return copied

n = _sync(RESULT_DIR_LOCAL, RESULT_DIR)
print(f'[Sync] {n} Datei(en) nach {RESULT_DIR} kopiert.')

if proc.returncode != 0:
    print('\n!! Training mit Fehler beendet. Letzte 40 Logzeilen:')
    with open(LOG_PATH) as f:
        raw = f.read().replace('\r', '\n').splitlines()
    for line in raw[-40:]:
        print(line)

In [ ]:
%cd /content

# WICHTIG: gsplat-Repo auf den TAG pinnen, der auch auf PyPI liegt.
# Sonst clonen wir 'main' (mit neueren Imports wie gsplat.color_correct),
# waehrend in Schritt 3 die aeltere PyPI-Version installiert wurde -> ImportError.
GSPLAT_VERSION = 'v1.5.3'

import shutil
# falls schon mal geclont (ggf. von main): wegwerfen und neu mit Tag clonen
if os.path.exists('/content/gsplat'):
    head = ''
    try:
        head = open('/content/gsplat/.git/HEAD').read().strip()
    except Exception:
        pass
    if GSPLAT_VERSION not in head:
        shutil.rmtree('/content/gsplat')

if not os.path.exists('/content/gsplat'):
    !git clone --depth 1 --branch {GSPLAT_VERSION} https://github.com/nerfstudio-project/gsplat.git

# WICHTIG: examples/datasets/ hat KEIN __init__.py -> es ist nur ein Namespace-
# Package. Colab hat aber das HuggingFace 'datasets' Paket vorinstalliert,
# und ein regulaeres Package gewinnt gegen ein Namespace-Package. Dadurch
# importiert simple_trainer.py das HF-datasets statt das lokale -> ImportError
# 'No module named datasets.colmap'. Fix: __init__.py-Dateien anlegen.
for sub in ['datasets']:
    init = f'/content/gsplat/examples/{sub}/__init__.py'
    if not os.path.exists(init):
        open(init, 'w').close()

%cd /content/gsplat/examples

# NICHT `pip install -r requirements.txt` -> die Datei enthaelt Pakete
# (nvidia-ncore, ppisp, fused-bilagrid, ein pycolmap-Fork mit Cython-Build),
# die unter Colab/T4 beim Wheel-Build crashen und fuer `default`-Training NICHT
# noetig sind. Stattdessen gezielt nur das installieren, was simple_trainer.py
# default zusaetzlich zu Schritt 3 braucht.
!pip install -q 'numpy<2.0.0' piexif tensorboard tensorly pyyaml \
    matplotlib scipy scikit-learn

# pycolmap-Fork (Pure-Python COLMAP-Reader; ANDERES Paket als PyPI 'pycolmap',
# wird vom gsplat-Beispiel-Dataloader importiert). Pure-Python -> kein CUDA-Build.
!pip install -q 'pycolmap @ git+https://github.com/rmbrualla/pycolmap@cc7ea4b7301720ac29287dbe450952511b32125e' \
    || echo 'WARNUNG: rmbrualla/pycolmap konnte nicht installiert werden - falls der Training-Start mit ImportError abbricht, bitte manuell nachinstallieren.'

# ============================================================================
# Robustness-Patches an den gsplat-Beispielen, alle idempotent.
# ============================================================================

# PATCH 1: datasets/colmap.py - KeyError vermeiden, wenn COLMAP Bilder registriert
# hat, die nicht (mehr) in {SCENE_DIR}/images liegen (z.B. nach erneutem Schritt 4
# mit veraenderter Frame-Auswahl gegenueber dem Drive-Cache).
colmap_py = '/content/gsplat/examples/datasets/colmap.py'
with open(colmap_py) as _f:
    _src = _f.read()
if '# patch_filter_missing_images' in _src:
    print('[SKIP] colmap.py bereits gepatcht.')
else:
    _needle = '        image_paths = [os.path.join(image_dir, colmap_to_image[f]) for f in image_names]'
    _insert = (
        '        # patch_filter_missing_images\n'
        '        _keep = [i for i, f in enumerate(image_names) if f in colmap_to_image]\n'
        '        if len(_keep) < len(image_names):\n'
        '            print(f"[Parser] {len(image_names) - len(_keep)} von COLMAP registrierte Bilder fehlen auf der Platte - werden uebersprungen.")\n'
        '            image_names = [image_names[i] for i in _keep]\n'
        '            camtoworlds = camtoworlds[_keep]\n'
        '            camera_ids = [camera_ids[i] for i in _keep]\n'
    )
    if _needle in _src:
        with open(colmap_py, 'w') as _f:
            _f.write(_src.replace(_needle, _insert + _needle, 1))
        print('[OK] colmap.py gepatcht (filtert COLMAP-Bilder ohne Disk-Pendant).')
    else:
        print('[WARN] colmap.py Patch-Anker nicht gefunden - gsplat-Version geaendert?')

# PATCH 2: simple_trainer.py - DataLoader-Defaults Colab-Free-tauglich machen.
# Hintergrund:
#  - Colab Free hat 2 vCPUs. gsplat nutzt default 4 Worker, das deadlockt
#    zuverlaessig (DataLoader produziert keinen einzigen Batch). Auch 2 Worker
#    reichen empirisch schon, um in der Praxis haengen zu bleiben - wir gehen
#    direkt auf 0 (synchrones Laden im Main-Process). Bei 100-200 kleinen JPGs
#    ist der Performance-Verlust irrelevant; voriger erfolgreicher Lauf lief
#    mit 0 Workern bei ~45 it/s.
#  - persistent_workers=True kollidiert mit num_workers=0 -> ValueError.
#  - pin_memory=True frisst RAM ohne Speed-Vorteil bei num_workers=0.
trainer_py = '/content/gsplat/examples/simple_trainer.py'
with open(trainer_py) as _f:
    _src = _f.read()
import re as _re
_patched_any = False
_new = _re.sub(r'num_workers\s*=\s*\d+', 'num_workers=0', _src)
if _new != _src:
    _patched_any = True
    _src = _new
if 'persistent_workers=True' in _src:
    _src = _src.replace('persistent_workers=True', 'persistent_workers=False')
    _patched_any = True
if 'pin_memory=True' in _src:
    _src = _src.replace('pin_memory=True', 'pin_memory=False')
    _patched_any = True
if _patched_any:
    with open(trainer_py, 'w') as _f:
        _f.write(_src)
    print('[OK] simple_trainer.py gepatcht (num_workers=0, persistent_workers=False, pin_memory=False).')
else:
    print('[SKIP] simple_trainer.py bereits gepatcht oder andere Defaults.')
!grep -n -E 'num_workers|persistent_workers|pin_memory' /content/gsplat/examples/simple_trainer.py

In [ ]:
import subprocess, time, shutil

# WICHTIG: Training schreibt seine Ckpts/Videos LOKAL (/content/...), NICHT direkt
# nach Drive. Gruende:
#  1) Drive-I/O ist langsam und kann waehrend Training haengen.
#  2) Bei Drive >75% voll fuehren Writes zu I/O-Errors und killen das Training.
# Am Ende der Zelle synchronisieren wir LOCAL -> Drive (auch bei Crash, damit
# Teil-Ergebnisse nicht verloren gehen). Die nachfolgenden Zellen lesen weiter
# ueber RESULT_DIR -> Drive-Pfad.
RESULT_DIR_LOCAL = '/content/gsplat_base'
RESULT_DIR       = f'{DRIVE_OUTPUT}/gsplat_base'
LOG_PATH         = '/content/train.log'
os.makedirs(RESULT_DIR_LOCAL, exist_ok=True)
os.makedirs(RESULT_DIR,       exist_ok=True)

# Training im Hintergrund + heartbeat alle 60s (nur letzte 5 Logzeilen).
# Verhindert "something went wrong displaying the page" / WebSocket-Reconnect-
# Loops durch tqdm-Output-Flut.
#
# Wichtige Trainings-Flags:
#  --strategy.refine-stop-iter 4000  -> Densification stoppt bei Step 4000,
#       danach werden nur noch existierende Gaussians verfeinert. Verhindert,
#       dass die GS-Zahl exponentiell ueber >500k wandert und CPU/GPU-OOM
#       ausloest. 4000 ist ein guter Tradeoff (Detailtiefe gut, Speicher OK).
#  --disable_viewer                  -> kein Viser-Server (in Colab eh blockiert).
env = {**os.environ, 'CUDA_VISIBLE_DEVICES': '0'}
log_f = open(LOG_PATH, 'w')
proc = subprocess.Popen(
    ['python', '/content/gsplat/examples/simple_trainer.py', 'default',
     '--data_dir',    SCENE_DIR,
     '--data_factor', str(DATA_FACTOR),
     '--result_dir',  RESULT_DIR_LOCAL,
     '--max_steps',   str(MAX_STEPS),
     '--strategy.refine-stop-iter', '4000',
     '--disable_viewer'],
    stdout=log_f, stderr=subprocess.STDOUT, env=env,
)

print(f'Training laeuft (PID {proc.pid}). Volles Log: {LOG_PATH}')
print(f'Lokales Ergebnis: {RESULT_DIR_LOCAL}  (wird am Ende nach Drive kopiert)')
print('Heartbeat alle 60s mit den letzten Logzeilen...\n')

try:
    while proc.poll() is None:
        time.sleep(60)
        try:
            with open(LOG_PATH, 'rb') as f:
                raw = f.read().decode('utf-8', errors='replace')
            lines = [ln for ln in raw.replace('\r', '\n').splitlines() if ln.strip()]
            print('--- ' + time.strftime('%H:%M:%S') + f' (PID {proc.pid}) ---')
            for line in lines[-5:]:
                print(line)
            print()
        except Exception as e:
            print('  (Log nicht lesbar:', e, ')')
except KeyboardInterrupt:
    print('\nAbbruch durch User -> terminiere Training...')
    proc.terminate()
    proc.wait(timeout=30)
    raise
finally:
    log_f.close()

print('Training beendet (Exit-Code:', proc.returncode, ')')

# Ergebnisse nach Drive synchronisieren - auch wenn Training gecrasht ist,
# damit zumindest die vorhandenen Checkpoints/Videos persistent gesichert sind.
def _sync(src_root, dst_root):
    copied = 0
    for sub in ('ckpts', 'videos', 'stats'):
        src = os.path.join(src_root, sub)
        if not os.path.isdir(src):
            continue
        dst = os.path.join(dst_root, sub)
        os.makedirs(dst, exist_ok=True)
        for fn in os.listdir(src):
            sp, dp = os.path.join(src, fn), os.path.join(dst, fn)
            if os.path.isfile(sp) and (not os.path.exists(dp)
                                       or os.path.getsize(sp) != os.path.getsize(dp)):
                shutil.copy(sp, dp); copied += 1
    return copied

n = _sync(RESULT_DIR_LOCAL, RESULT_DIR)
print(f'[Sync] {n} Datei(en) nach {RESULT_DIR} kopiert.')

if proc.returncode != 0:
    print('\n!! Training mit Fehler beendet. Letzte 40 Logzeilen:')
    with open(LOG_PATH) as f:
        raw = f.read().replace('\r', '\n').splitlines()
    for line in raw[-40:]:
        print(line)

## Schritt 8: Flythrough-Video anzeigen

gsplat rendert am Ende des Trainings automatisch einen Flythrough entlang einer interpolierten Trajektorie.

In [ ]:
from IPython.display import HTML, display
from base64 import b64encode
import glob

videos = sorted(glob.glob(f'{RESULT_DIR}/videos/*.mp4'))
print('Gefundene Videos:')
for v in videos:
    print(' ', v)

if not videos:
    print('Keine Videos gefunden. Pruefe den Trainings-Output unter', RESULT_DIR)
else:
    latest = videos[-1]
    mp4 = open(latest, 'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
    display(HTML(f'<h4>{os.path.basename(latest)}</h4>'
                 f'<video width=720 controls loop autoplay muted>'
                 f'<source src="{data_url}" type="video/mp4"></video>'))

## Schritt 8.5: Als `.ply` exportieren (fuer SuperSplat & Co.)

Der `.pt` Checkpoint ist PyTorch-spezifisch. Diese Zelle konvertiert ihn ins standardkompatible **3DGS `.ply` Format**, das du in folgende Tools laden kannst:

- [SuperSplat](https://playcanvas.com/supersplat/editor) (Browser, drag & drop)
- [antimatter15 WebGL Viewer](https://antimatter15.com/splat/)
- nerfstudio (`ns-viewer` mit splatfacto)
- Unity / Unreal Plugins fuer 3DGS

Die `.ply` landet direkt in deinem Drive.

In [ ]:
import torch
import numpy as np
from plyfile import PlyData, PlyElement

def export_gsplat_ckpt_to_ply(ckpt_path: str, ply_path: str):
    """Konvertiert ein gsplat .pt Checkpoint ins standard 3DGS .ply Format."""
    ckpt = torch.load(ckpt_path, map_location='cpu')
    splats = ckpt['splats']

    xyz = splats['means'].numpy()
    N = xyz.shape[0]
    normals = np.zeros_like(xyz)

    sh0 = splats['sh0'].numpy()                       # [N, 1, 3]
    shN = splats['shN'].numpy()                       # [N, K, 3]
    f_dc = sh0.reshape(N, -1)                         # [N, 3]
    f_rest = shN.transpose(0, 2, 1).reshape(N, -1)    # [N, 3*K]

    opacities = splats['opacities'].numpy().reshape(N, 1)
    scales = splats['scales'].numpy()                 # log-space, wird so erwartet
    rots = splats['quats'].numpy()                    # [w, x, y, z]

    dtype_full = [('x', 'f4'), ('y', 'f4'), ('z', 'f4'),
                  ('nx', 'f4'), ('ny', 'f4'), ('nz', 'f4')]
    for i in range(f_dc.shape[1]):
        dtype_full.append((f'f_dc_{i}', 'f4'))
    for i in range(f_rest.shape[1]):
        dtype_full.append((f'f_rest_{i}', 'f4'))
    dtype_full += [('opacity', 'f4'),
                   ('scale_0', 'f4'), ('scale_1', 'f4'), ('scale_2', 'f4'),
                   ('rot_0', 'f4'), ('rot_1', 'f4'), ('rot_2', 'f4'), ('rot_3', 'f4')]

    elements = np.empty(N, dtype=dtype_full)
    attributes = np.concatenate((xyz, normals, f_dc, f_rest, opacities, scales, rots), axis=1)
    elements[:] = list(map(tuple, attributes))

    PlyData([PlyElement.describe(elements, 'vertex')]).write(ply_path)
    size_mb = os.path.getsize(ply_path) / 1024 / 1024
    print(f'[OK] {N:,} Gaussians exportiert -> {ply_path} ({size_mb:.1f} MB)')

# Baseline-Checkpoint exportieren
ckpt = sorted(glob.glob(f'{RESULT_DIR}/ckpts/*.pt'))[-1]
ply_out = f'{DRIVE_OUTPUT}/room.ply'
export_gsplat_ckpt_to_ply(ckpt, ply_out)

print('\nLade jetzt die Datei aus Drive runter:')
print(' ', ply_out)
print('und ziehe sie in https://playcanvas.com/supersplat/editor')

## Schritt 9 (optional): Difix3D - Artefakte entfernen

Wenn du im Flythrough sichtbare Floater, Schlieren oder Blur siehst (besonders an Stellen, wo du im Video **nicht** warst), kannst du das Modell mit Difix3D weiter verfeinern.

**Achtung:** Difix3D laedt zusaetzlich SD-Turbo (~5 GB) in den VRAM. Auf T4 ist das knapp - falls OOM:
- `DATA_FACTOR = 8` in Schritt 2 setzen
- weniger Frames (`TARGET_FRAMES = 100`)

In [ ]:
RUN_DIFIX3D = True  # auf False setzen wenn du es ueberspringen willst

if RUN_DIFIX3D:
    %cd /content
    if not os.path.exists('/content/Difix3D'):
        !git clone --depth 1 https://github.com/nv-tlabs/Difix3D.git
    %cd /content/Difix3D
    !pip install -q -r requirements.txt

    # HF Cache nach Drive damit es nicht jedes Mal neu laedt
    os.environ['HF_HOME'] = '/content/drive/MyDrive/room3d/hf_cache'
    os.makedirs(os.environ['HF_HOME'], exist_ok=True)

In [ ]:
if RUN_DIFIX3D:
    DIFIX_RESULT = f'{DRIVE_OUTPUT}/difix3d'
    CKPT = sorted(glob.glob(f'{RESULT_DIR}/ckpts/*.pt'))[-1]
    print('Verwende Checkpoint:', CKPT)

    !cd /content/Difix3D && CUDA_VISIBLE_DEVICES=0 python examples/gsplat/simple_trainer_difix3d.py default \
        --data_dir {SCENE_DIR} \
        --data_factor {DATA_FACTOR} \
        --result_dir {DIFIX_RESULT} \
        --no-normalize-world-space \
        --test_every 1 \
        --ckpt {CKPT}

In [ ]:
# Difix3D Video anzeigen + .ply exportieren
if RUN_DIFIX3D:
    videos = sorted(glob.glob(f'{DIFIX_RESULT}/videos/*.mp4'))
    if videos:
        latest = videos[-1]
        mp4 = open(latest, 'rb').read()
        data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
        display(HTML(f'<h4>Difix3D Ergebnis: {os.path.basename(latest)}</h4>'
                     f'<video width=720 controls loop autoplay muted>'
                     f'<source src="{data_url}" type="video/mp4"></video>'))
    else:
        print('Keine Difix3D-Videos gefunden unter', DIFIX_RESULT)

    # Difix3D-Checkpoint auch als .ply exportieren
    difix_ckpts = sorted(glob.glob(f'{DIFIX_RESULT}/ckpts/*.pt'))
    if difix_ckpts:
        ply_out_difix = f'{DRIVE_OUTPUT}/room_difix3d.ply'
        export_gsplat_ckpt_to_ply(difix_ckpts[-1], ply_out_difix)

## Outputs in deinem Drive

Alles persistent in `MyDrive/room3d/output/`:

```
output/
  room.ply                        <-- DAS hier in SuperSplat reinziehen
  room_difix3d.ply                (falls Schritt 9 lief)
  gsplat_base/
    ckpts/ckpt_29999_rank0.pt    <- PyTorch Checkpoint (Backup)
    videos/                       <- Flythrough Videos
    stats/                        <- Metriken
  difix3d/                       (falls Schritt 9 lief)
    ckpts/...
    videos/...
```

## So schaust du dein Modell an

1. **Browser (am einfachsten):** [SuperSplat](https://playcanvas.com/supersplat/editor) -> `room.ply` per Drag&Drop reinziehen -> mit Maus durchs Zimmer fliegen.
2. **Browser (Alternative):** [antimatter15 viewer](https://antimatter15.com/splat/) -> File hochladen.
3. **Lokal mit nerfstudio:** `pip install nerfstudio` -> ueber `splatfacto` Loader importieren -> `ns-viewer`.

---

## Troubleshooting

| Problem | Fix |
|---|---|
| `COLMAP konnte keine Rekonstruktion erstellen` | Video hat zu wenig Overlap/Textur. Langsamer filmen, mehr Objekte im Bild. |
| `CUDA out of memory` waehrend Training | `DATA_FACTOR=8`, `TARGET_FRAMES=100` |
| Colab disconnected mitten im Training | Browser-Tab fokussiert lassen. Checkpoints liegen in Drive -> spaeter mit gleichem `RESULT_DIR` weitermachen. |
| Sehr verschwommenes Ergebnis | Mehr Frames, hoehere Aufloesung (`DATA_FACTOR=2`), oder Difix3D-Schritt anwenden |
| `.ply` Export schlaegt fehl mit KeyError 'splats' | gsplat-Version anders strukturiert. Inhalt anzeigen mit: `print(torch.load(ckpt, map_location='cpu').keys())` |
| SuperSplat zeigt nichts an | `.ply` zu gross (>500 MB)? Erst runterladen, dann hochladen, nicht direkt aus Drive einbinden. |